In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/NLP_Research"

folders = [
    "datasets",
    "checkpoints",
    "reports",
    "logs"
]

for folder in folders:
    os.makedirs(
        os.path.join(PROJECT_DIR, folder),
        exist_ok=True
    )

print("Project directory:", PROJECT_DIR)

for folder in folders:
    print(folder, "created")

Project directory: /content/drive/MyDrive/NLP_Research
datasets created
checkpoints created
reports created
logs created


In [ ]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3,
            2
        ),
        "GB"
    )

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4
GPU memory: 14.56 GB


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import os

CHECKPOINT_DIR = (
    "/content/drive/MyDrive/NLP_Research/checkpoints"
)

for filename in os.listdir(CHECKPOINT_DIR):
    path = os.path.join(CHECKPOINT_DIR, filename)

    print(
        filename,
        round(os.path.getsize(path) / (1024**2), 2),
        "MB"
    )

model_metadata.json 0.0 MB
best_model.pt 417.73 MB


In [ ]:
import json

METADATA_PATH = (
    "/content/drive/MyDrive/NLP_Research/checkpoints/"
    "model_metadata.json"
)

with open(METADATA_PATH, "r") as f:
    metadata = json.load(f)

print(json.dumps(metadata, indent=4))

{
    "model_name": "bert-base-uncased",
    "max_length": 256,
    "best_epoch": 3,
    "random_seed": 42,
    "label_names": [
        "Neutral",
        "IdentityDirectedAbuse",
        "AffiliationDirectedAbuse",
        "PersonDirectedAbuse",
        "CounterSpeech"
    ],
    "input": "current_comment_only"
}


In [ ]:
import torch

CHECKPOINT_PATH = (
    "/content/drive/MyDrive/NLP_Research/checkpoints/"
    "best_model.pt"
)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
    weights_only=False
)

print("Checkpoint type:", type(checkpoint))

if isinstance(checkpoint, dict):
    print("\nCheckpoint keys:")
    for key in checkpoint.keys():
        print("-", key)

Checkpoint type: <class 'dict'>

Checkpoint keys:
- model_state_dict
- model_name
- best_epoch
- best_dev_macro_f1_threshold_0_5
- max_length
- label_names
- random_seed


In [ ]:
state_dict = checkpoint["model_state_dict"]

print("Number of parameters stored:", len(state_dict))

print("\nModel parameter names and shapes:")

for name, tensor in state_dict.items():
    print(f"{name:65s} {tuple(tensor.shape)}")

Number of parameters stored: 201

Model parameter names and shapes:
bert.embeddings.word_embeddings.weight                            (30522, 768)
bert.embeddings.position_embeddings.weight                        (512, 768)
bert.embeddings.token_type_embeddings.weight                      (2, 768)
bert.embeddings.LayerNorm.weight                                  (768,)
bert.embeddings.LayerNorm.bias                                    (768,)
bert.encoder.layer.0.attention.self.query.weight                  (768, 768)
bert.encoder.layer.0.attention.self.query.bias                    (768,)
bert.encoder.layer.0.attention.self.key.weight                    (768, 768)
bert.encoder.layer.0.attention.self.key.bias                      (768,)
bert.encoder.layer.0.attention.self.value.weight                  (768, 768)
bert.encoder.layer.0.attention.self.value.bias                    (768,)
bert.encoder.layer.0.attention.output.dense.weight                (768, 768)
bert.encoder.layer.0.attenti

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

MODEL_NAME = "bert-base-uncased"
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class BertTextClassifier(nn.Module):

    def __init__(self, model_name, num_labels=5, dropout=0.1):
        super().__init__()

        self.bert = AutoModel.from_pretrained(
            model_name,
            add_pooling_layer=True
        )

        self.dropout = nn.Dropout(dropout)

        self.classifier = nn.Linear(
            self.bert.config.hidden_size,
            num_labels
        )

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[:, 0, :]

        cls_output = self.dropout(cls_output)

        logits = self.classifier(cls_output)

        return logits


# Initialize architecture
e2_model = BertTextClassifier(
    model_name=MODEL_NAME,
    num_labels=5
)

# Load trained E2 weights
e2_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

e2_model.to(DEVICE)
e2_model.eval()

print("E2 model loaded successfully!")
print("Device:", DEVICE)
print("Parameters:", sum(p.numel() for p in e2_model.parameters()))

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


E2 model loaded successfully!
Device: cuda
Parameters: 109486085


In [ ]:
import os
import torch

ENCODER_PATH = (
    "/content/drive/MyDrive/NLP_Research/checkpoints/"
    "e2_bert_encoder.pt"
)

torch.save(
    e2_model.bert.state_dict(),
    ENCODER_PATH
)

print("E2 encoder saved successfully!")
print("Location:", ENCODER_PATH)

E2 encoder saved successfully!
Location: /content/drive/MyDrive/NLP_Research/checkpoints/e2_bert_encoder.pt


In [ ]:
import os

size_mb = os.path.getsize(ENCODER_PATH) / (1024 ** 2)

print("Encoder exists:", os.path.exists(ENCODER_PATH))
print("Encoder size:", round(size_mb, 2), "MB")

Encoder exists: True
Encoder size: 417.72 MB


In [ ]:
import os
import shutil

SOURCE_DIR = "datasets/processed"
DEST_DIR = "/content/drive/MyDrive/NLP_Research/datasets"

os.makedirs(DEST_DIR, exist_ok=True)

for split in ["train", "dev", "test"]:

    filename = f"e3_{split}_pairs.csv"

    source = os.path.join(SOURCE_DIR, filename)
    destination = os.path.join(DEST_DIR, filename)

    assert os.path.exists(source), f"Missing: {source}"

    shutil.copy2(source, destination)

    print(f"Saved: {filename}")

print("\nAll E3 datasets backed up to Google Drive.")

AssertionError: Missing: datasets/processed/e3_train_pairs.csv

In [ ]:
import os

BASE = "/content/drive/MyDrive/NLP_Research"

for folder, subfolders, files in os.walk(BASE):
    for filename in files:
        if filename.endswith(".csv"):
            print(os.path.join(folder, filename))

/content/drive/MyDrive/NLP_Research/datasets/cad_test_processed.csv
/content/drive/MyDrive/NLP_Research/datasets/cad_dev_processed.csv
/content/drive/MyDrive/NLP_Research/datasets/cad_train_processed.csv


In [ ]:
import os

for filename in [
    "cad_train_processed.csv",
    "cad_dev_processed.csv",
    "cad_test_processed.csv"
]:
    path = f"datasets/processed/{filename}"
    print(filename, os.path.exists(path))

cad_train_processed.csv False
cad_dev_processed.csv False
cad_test_processed.csv False


In [ ]:
import os
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/NLP_Research/datasets"

train_df = pd.read_csv(
    os.path.join(DATA_DIR, "cad_train_processed.csv"),
    keep_default_na=False
)

dev_df = pd.read_csv(
    os.path.join(DATA_DIR, "cad_dev_processed.csv"),
    keep_default_na=False
)

test_df = pd.read_csv(
    os.path.join(DATA_DIR, "cad_test_processed.csv"),
    keep_default_na=False
)

datasets = {
    "train": train_df,
    "dev": dev_df,
    "test": test_df
}

for name, df in datasets.items():
    print(f"{name.upper()}: {len(df)} records")

TRAIN: 13584 records
DEV: 4526 records
TEST: 5307 records


In [ ]:
import os
import pandas as pd

OUTPUT_DIR = os.path.join(DATA_DIR, "e3_pairs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

e3_datasets = {}

required_columns = [
    "id", "text", "parent_id", "parent_text",
    "labels", "label_vector",
    "label_0", "label_1", "label_2", "label_3", "label_4",
    "split"
]

total_pairs = 0

for split_name, df in datasets.items():

    id_to_text = dict(zip(df["id"], df["text"]))

    reconstructed_parent = df["parent_id"].map(id_to_text)

    has_parent_id = ~df["parent_id"].isin(["", "None"])

    valid_mask = (
        has_parent_id
        & reconstructed_parent.notna()
        & reconstructed_parent.eq(df["parent_text"])
    )

    paired_df = df.loc[valid_mask].copy()

    paired_df["parent_text"] = reconstructed_parent.loc[valid_mask]

    paired_df = paired_df[required_columns].reset_index(drop=True)

    output_path = os.path.join(
        OUTPUT_DIR,
        f"e3_{split_name}_pairs.csv"
    )

    paired_df.to_csv(output_path, index=False)

    e3_datasets[split_name] = paired_df
    total_pairs += len(paired_df)

    print(f"{split_name.upper()}: {len(paired_df)} pairs saved")

print("\nTOTAL PAIRS:", total_pairs)
print("Saved permanently in:", OUTPUT_DIR)

TRAIN: 7571 pairs saved
DEV: 2390 pairs saved
TEST: 2875 pairs saved

TOTAL PAIRS: 12836
Saved permanently in: /content/drive/MyDrive/NLP_Research/datasets/e3_pairs


In [ ]:
for split in ["train", "dev", "test"]:

    path = os.path.join(
        OUTPUT_DIR,
        f"e3_{split}_pairs.csv"
    )

    print(
        f"{split}:",
        os.path.exists(path),
        f"{os.path.getsize(path)/(1024**2):.2f} MB"
    )

train: True 3.18 MB
dev: True 1.13 MB
test: True 1.35 MB


In [ ]:
for split_name, df in e3_datasets.items():

    print(f"\n{split_name.upper()}")

    print("Label vector example:")
    print(repr(df["label_vector"].iloc[0]))

    print("Label type:")
    print(type(df["label_vector"].iloc[0]))

    print("Current comment:")
    print(df["text"].iloc[0])

    print("Parent comment:")
    print(df["parent_text"].iloc[0])


TRAIN
Label vector example:
'[1, 0, 0, 0, 0]'
Label type:
<class 'str'>
Current comment:
No, I like binging the big bang theory instead. Bazoople!
Parent comment:
Literally three days old. No new developments in this story. Are you not addicted to news media or something?

DEV
Label vector example:
'[1, 0, 0, 0, 0]'
Label type:
<class 'str'>
Current comment:
Someone who doesn't think you need dysphoria to be transgender
Parent comment:
What the hell is a tucute?

TEST
Label vector example:
'[1, 0, 0, 0, 0]'
Label type:
<class 'str'>
Current comment:
he's worth around 100 million dollars, he doesn't really need to make more money. although i agree, still very odd though
Parent comment:
The Tom DeLonge one is weird because he could almost certainly make more money in music than he has in TTS. And yet the whole thing is just bizarre.. I don't trust him at all but I don't understand the motive behind it.


In [ ]:
import torch
from transformers import AutoTokenizer

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(
    "bert-base-uncased",
    token=False
)

e2_model.eval()

print("Device:", DEVICE)
print("Model device:", next(e2_model.parameters()).device)
print("Tokenizer:", tokenizer.name_or_path)
print("Maximum length:", 256)

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

Device: cuda
Model device: cuda:0
Tokenizer: bert-base-uncased
Maximum length: 256


In [ ]:

import os
import ast
import json
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    accuracy_score
)

# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

MAX_LENGTH = 256
BATCH_SIZE = 32

BASE_DIR = "/content/drive/MyDrive/NLP_Research"

REPORT_DIR = os.path.join(BASE_DIR, "reports", "e2_subset")
os.makedirs(REPORT_DIR, exist_ok=True)

LABEL_NAMES = [
    "Neutral",
    "IdentityDirectedAbuse",
    "AffiliationDirectedAbuse",
    "PersonDirectedAbuse",
    "CounterSpeech"
]

print("Device:", DEVICE)
print("Batch size:", BATCH_SIZE)


# --------------------------------------------------
# 2. Convert label strings into numerical arrays
# --------------------------------------------------

def parse_labels(label_string):
    labels = ast.literal_eval(label_string)

    labels = np.asarray(labels, dtype=np.float32)

    assert labels.shape == (5,), "Incorrect label dimensions"

    return labels


# --------------------------------------------------
# 3. Create inference dataset
# --------------------------------------------------

class E2InferenceDataset(Dataset):

    def __init__(self, dataframe, tokenizer):

        self.texts = dataframe["text"].tolist()

        self.labels = np.stack([
            parse_labels(x)
            for x in dataframe["label_vector"]
        ])

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        return {
            "text": self.texts[index],
            "labels": self.labels[index]
        }


# --------------------------------------------------
# 4. Batch tokenization
# --------------------------------------------------

def collate_fn(batch):

    texts = [item["text"] for item in batch]

    labels = torch.tensor(
        np.stack([item["labels"] for item in batch]),
        dtype=torch.float32
    )

    encoded = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    return {
        "input_ids": encoded["input_ids"],
        "attention_mask": encoded["attention_mask"],
        "labels": labels
    }


# --------------------------------------------------
# 5. Generate predictions
# --------------------------------------------------

def generate_predictions(dataframe, split_name):

    dataset = E2InferenceDataset(dataframe, tokenizer)

    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        collate_fn=collate_fn,
        num_workers=0
    )

    all_probabilities = []
    all_labels = []

    e2_model.eval()

    with torch.inference_mode():

        for batch_number, batch in enumerate(loader):

            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)

            logits = e2_model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            probabilities = torch.sigmoid(logits)

            all_probabilities.append(
                probabilities.cpu().numpy()
            )

            all_labels.append(
                batch["labels"].numpy()
            )

            if (batch_number + 1) % 20 == 0:
                print(
                    f"{split_name}: "
                    f"{batch_number + 1}/{len(loader)} batches completed"
                )

    probabilities = np.concatenate(all_probabilities)
    labels = np.concatenate(all_labels)

    print(f"\n{split_name} inference completed!")
    print("Predictions shape:", probabilities.shape)
    print("Labels shape:", labels.shape)

    return probabilities, labels


# --------------------------------------------------
# 6. Generate DEV and TEST predictions
# --------------------------------------------------

dev_probs, dev_labels = generate_predictions(
    e3_datasets["dev"],
    "DEV"
)

test_probs, test_labels = generate_predictions(
    e3_datasets["test"],
    "TEST"
)

print("\nBoth inference runs completed successfully.")


Device: cuda
Batch size: 32
DEV: 20/75 batches completed
DEV: 40/75 batches completed
DEV: 60/75 batches completed

DEV inference completed!
Predictions shape: (2390, 5)
Labels shape: (2390, 5)
TEST: 20/90 batches completed
TEST: 40/90 batches completed
TEST: 60/90 batches completed
TEST: 80/90 batches completed

TEST inference completed!
Predictions shape: (2875, 5)
Labels shape: (2875, 5)

Both inference runs completed successfully.


In [ ]:

# --------------------------------------------------
# 7. Find best threshold for each label using DEV
# --------------------------------------------------

thresholds = {}

threshold_values = np.arange(0.05, 0.951, 0.01)

for i, label_name in enumerate(LABEL_NAMES):

    best_f1 = -1
    best_threshold = 0.5

    for threshold in threshold_values:

        predictions = (
            dev_probs[:, i] >= threshold
        ).astype(int)

        score = f1_score(
            dev_labels[:, i],
            predictions,
            zero_division=0
        )

        if score > best_f1:
            best_f1 = score
            best_threshold = threshold

    thresholds[label_name] = float(best_threshold)

    print(
        f"{label_name:30s} "
        f"Threshold: {best_threshold:.2f} "
        f"Dev F1: {best_f1:.4f}"
    )


# Save thresholds
threshold_path = os.path.join(
    REPORT_DIR,
    "e2_subset_thresholds.json"
)

with open(threshold_path, "w") as f:
    json.dump(thresholds, f, indent=4)

print("\nThresholds saved:", threshold_path)


Neutral                        Threshold: 0.11 Dev F1: 0.9139
IdentityDirectedAbuse          Threshold: 0.13 Dev F1: 0.4450
AffiliationDirectedAbuse       Threshold: 0.14 Dev F1: 0.5315
PersonDirectedAbuse            Threshold: 0.09 Dev F1: 0.4500
CounterSpeech                  Threshold: 0.07 Dev F1: 0.0541

Thresholds saved: /content/drive/MyDrive/NLP_Research/reports/e2_subset/e2_subset_thresholds.json


In [ ]:

# --------------------------------------------------
# 8. Apply frozen DEV thresholds to TEST
# --------------------------------------------------

threshold_array = np.array([
    thresholds[name] for name in LABEL_NAMES
])

test_predictions = (
    test_probs >= threshold_array
).astype(int)


# --------------------------------------------------
# 9. Calculate metrics
# --------------------------------------------------

macro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

micro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="micro",
    zero_division=0
)

macro_precision = precision_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

macro_recall = recall_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0
)

hamming = hamming_loss(
    test_labels,
    test_predictions
)

subset_accuracy = accuracy_score(
    test_labels,
    test_predictions
)


print("\n========== E2 SUBSET TEST RESULTS ==========")

print(f"Macro F1:        {macro_f1:.4f}")
print(f"Micro F1:        {micro_f1:.4f}")
print(f"Macro Precision: {macro_precision:.4f}")
print(f"Macro Recall:    {macro_recall:.4f}")
print(f"Hamming Loss:    {hamming:.4f}")
print(f"Subset Accuracy: {subset_accuracy:.4f}")


# --------------------------------------------------
# 10. Per-label performance
# --------------------------------------------------

print("\n========== PER-LABEL RESULTS ==========")

per_label_f1 = f1_score(
    test_labels,
    test_predictions,
    average=None,
    zero_division=0
)

for name, score in zip(LABEL_NAMES, per_label_f1):

    print(f"{name:30s}: {score:.4f}")


# --------------------------------------------------
# 11. Save metrics and predictions
# --------------------------------------------------

metrics = {
    "experiment": "E2_on_E3_subset",
    "test_records": int(len(test_labels)),
    "thresholds": thresholds,
    "macro_f1": float(macro_f1),
    "micro_f1": float(micro_f1),
    "macro_precision": float(macro_precision),
    "macro_recall": float(macro_recall),
    "hamming_loss": float(hamming),
    "subset_accuracy": float(subset_accuracy),
    "per_label_f1": {
        name: float(score)
        for name, score in zip(LABEL_NAMES, per_label_f1)
    }
}

with open(
    os.path.join(REPORT_DIR, "e2_subset_metrics.json"),
    "w"
) as f:
    json.dump(metrics, f, indent=4)


# Save record-level predictions
prediction_df = e3_datasets["test"][
    ["id", "text", "label_vector"]
].copy()

for i, name in enumerate(LABEL_NAMES):

    prediction_df[f"{name}_probability"] = test_probs[:, i]

    prediction_df[f"{name}_prediction"] = test_predictions[:, i]

prediction_df.to_csv(
    os.path.join(REPORT_DIR, "e2_subset_test_predictions.csv"),
    index=False
)

print("\nE2 subset results saved successfully!")
print("Location:", REPORT_DIR)



========== E2 SUBSET TEST RESULTS ==========
Macro F1:        0.4413
Micro F1:        0.8242
Macro Precision: 0.4280
Macro Recall:    0.4578
Hamming Loss:    0.0751
Subset Accuracy: 0.7770

========== PER-LABEL RESULTS ==========
Neutral                       : 0.9140
IdentityDirectedAbuse         : 0.4596
AffiliationDirectedAbuse      : 0.3930
PersonDirectedAbuse           : 0.4397
CounterSpeech                 : 0.0000

E2 subset results saved successfully!
Location: /content/drive/MyDrive/NLP_Research/reports/e2_subset


In [ ]:

import torch
import torch.nn as nn

class ContextAwareBERT(nn.Module):

    def __init__(
        self,
        shared_encoder,
        hidden_size=768,
        num_labels=5,
        fusion_dropout=0.2
    ):
        super().__init__()

        # Reuse the trained E2 BERT encoder
        self.bert = shared_encoder

        # Fusion representation:
        # hC + hP + |hC-hP| + hC*hP
        fusion_size = hidden_size * 4

        self.classifier = nn.Sequential(

            nn.Linear(fusion_size, 256),

            nn.ReLU(),

            nn.Dropout(fusion_dropout),

            nn.Linear(256, num_labels)
        )

    def encode(self, input_ids, attention_mask):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        # Extract [CLS] representation
        return outputs.last_hidden_state[:, 0, :]

    def forward(
        self,
        current_input_ids,
        current_attention_mask,
        parent_input_ids,
        parent_attention_mask
    ):

        # Encode current comment
        h_current = self.encode(
            current_input_ids,
            current_attention_mask
        )

        # Encode parent comment using the SAME encoder
        h_parent = self.encode(
            parent_input_ids,
            parent_attention_mask
        )

        # Feature fusion
        difference = torch.abs(h_current - h_parent)

        product = h_current * h_parent

        fused = torch.cat(
            [
                h_current,
                h_parent,
                difference,
                product
            ],
            dim=1
        )

        # Five classification logits
        logits = self.classifier(fused)

        return logits


# Initialize E3 using the existing E2 BERT encoder
e3_model = ContextAwareBERT(
    shared_encoder=e2_model.bert,
    hidden_size=768,
    num_labels=5
)

e3_model = e3_model.to(DEVICE)

print("E3 model initialized successfully!")
print("Device:", next(e3_model.parameters()).device)
print("Fusion dimensions:", 768 * 4)
print("Classifier:", e3_model.classifier)


E3 model initialized successfully!
Device: cuda:0
Fusion dimensions: 3072
Classifier: Sequential(
  (0): Linear(in_features=3072, out_features=256, bias=True)
  (1): ReLU()
  (2): Dropout(p=0.2, inplace=False)
  (3): Linear(in_features=256, out_features=5, bias=True)
)


In [ ]:
e3_model.eval()

batch_size = 2
sequence_length = 32

dummy_current_ids = torch.randint(
    0, 30522,
    (batch_size, sequence_length),
    device=DEVICE
)

dummy_parent_ids = torch.randint(
    0, 30522,
    (batch_size, sequence_length),
    device=DEVICE
)

dummy_current_mask = torch.ones(
    batch_size, sequence_length,
    dtype=torch.long,
    device=DEVICE
)

dummy_parent_mask = torch.ones(
    batch_size, sequence_length,
    dtype=torch.long,
    device=DEVICE
)

with torch.inference_mode():

    output = e3_model(
        dummy_current_ids,
        dummy_current_mask,
        dummy_parent_ids,
        dummy_parent_mask
    )

print("Output shape:", output.shape)
print("Output device:", output.device)
print("E3 total parameters:",
      sum(p.numel() for p in e3_model.parameters()))

Output shape: torch.Size([2, 5])
Output device: cuda:0
E3 total parameters: 110270213


In [ ]:
import torch

torch.cuda.empty_cache()

free_memory, total_memory = torch.cuda.mem_get_info()

print("Total GPU memory:",
      round(total_memory / 1024**3, 2), "GB")

print("Free GPU memory:",
      round(free_memory / 1024**3, 2), "GB")

print("Allocated GPU memory:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Total GPU memory: 14.56 GB
Free GPU memory: 13.97 GB
Allocated GPU memory: 0.42 GB


In [ ]:

import os
import ast
import json
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import f1_score
from transformers import get_linear_schedule_with_warmup

# -------------------------------
# Configuration
# -------------------------------

SEED = 42
MAX_LENGTH = 256
BATCH_SIZE = 4
GRAD_ACCUMULATION = 2
EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

DEVICE = torch.device("cuda")

BASE_DIR = "/content/drive/MyDrive/NLP_Research"

DATA_DIR = os.path.join(BASE_DIR, "datasets", "e3_pairs")

CHECKPOINT_DIR = os.path.join(BASE_DIR, "checkpoints", "e3")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Training configuration initialized.")


# -------------------------------
# Load paired datasets
# -------------------------------

train_df = pd.read_csv(
    os.path.join(DATA_DIR, "e3_train_pairs.csv"),
    keep_default_na=False
)

dev_df = pd.read_csv(
    os.path.join(DATA_DIR, "e3_dev_pairs.csv"),
    keep_default_na=False
)

test_df = pd.read_csv(
    os.path.join(DATA_DIR, "e3_test_pairs.csv"),
    keep_default_na=False
)

print("Train:", len(train_df))
print("Dev:", len(dev_df))
print("Test:", len(test_df))


# -------------------------------
# Dataset class
# -------------------------------

def parse_label_vector(value):
    labels = np.asarray(
        ast.literal_eval(value),
        dtype=np.float32
    )

    assert labels.shape == (5,)

    return labels


class ContextDataset(Dataset):

    def __init__(self, dataframe):
        self.current_texts = dataframe["text"].tolist()
        self.parent_texts = dataframe["parent_text"].tolist()

        self.labels = [
            parse_label_vector(value)
            for value in dataframe["label_vector"]
        ]

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, index):
        return {
            "current_text": self.current_texts[index],
            "parent_text": self.parent_texts[index],
            "labels": self.labels[index]
        }


# -------------------------------
# Dynamic batch tokenization
# -------------------------------

def context_collate_fn(batch):

    current_texts = [
        item["current_text"] for item in batch
    ]

    parent_texts = [
        item["parent_text"] for item in batch
    ]

    labels = torch.tensor(
        np.stack([item["labels"] for item in batch]),
        dtype=torch.float32
    )

    current_tokens = tokenizer(
        current_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    parent_tokens = tokenizer(
        parent_texts,
        padding=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    return {
        "current_input_ids": current_tokens["input_ids"],
        "current_attention_mask": current_tokens["attention_mask"],
        "parent_input_ids": parent_tokens["input_ids"],
        "parent_attention_mask": parent_tokens["attention_mask"],
        "labels": labels
    }


# -------------------------------
# Create DataLoaders
# -------------------------------

train_dataset = ContextDataset(train_df)
dev_dataset = ContextDataset(dev_df)
test_dataset = ContextDataset(test_df)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=context_collate_fn,
    num_workers=0,
    pin_memory=True
)

dev_loader = DataLoader(
    dev_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=context_collate_fn,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False,
    collate_fn=context_collate_fn,
    num_workers=0,
    pin_memory=True
)

print("\nDataLoaders ready!")
print("Training batches:", len(train_loader))
print("Validation batches:", len(dev_loader))
print("Effective batch size:", BATCH_SIZE * GRAD_ACCUMULATION)


Training configuration initialized.
Train: 7571
Dev: 2390
Test: 2875

DataLoaders ready!
Training batches: 1893
Validation batches: 299
Effective batch size: 8


In [ ]:
import os

print("Drive available:", os.path.exists(BASE_DIR))
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Free GPU memory:",
      round(torch.cuda.mem_get_info()[0] / 1024**3, 2), "GB")

In [ ]:

import os
import json
import math
import time
import numpy as np
import torch
import torch.nn as nn

from torch.optim import AdamW
from sklearn.metrics import f1_score
from transformers import get_linear_schedule_with_warmup

# ==================================================
# 1. CONFIGURATION
# ==================================================

EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
GRAD_CLIP = 1.0
PATIENCE = 2
SAVE_EVERY_UPDATES = 100

ACCUMULATION = GRAD_ACCUMULATION

CHECKPOINT_DIR = os.path.join(
    BASE_DIR, "checkpoints", "e3"
)

REPORT_DIR = os.path.join(
    BASE_DIR, "reports", "e3"
)

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

BEST_MODEL_PATH = os.path.join(
    CHECKPOINT_DIR, "best_model.pt"
)

LATEST_PATH = os.path.join(
    CHECKPOINT_DIR, "latest_checkpoint.pt"
)

HISTORY_PATH = os.path.join(
    REPORT_DIR, "training_history.json"
)

# Reproducibility and GPU settings
torch.backends.cuda.matmul.allow_tf32 = True

criterion = nn.BCEWithLogitsLoss()

optimizer = AdamW(
    e3_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

updates_per_epoch = math.ceil(
    len(train_loader) / ACCUMULATION
)

total_training_updates = updates_per_epoch * EPOCHS

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_training_updates),
    num_training_steps=total_training_updates
)

# FP16 gradient scaler
scaler = torch.amp.GradScaler("cuda")

print("Updates per epoch:", updates_per_epoch)
print("Total optimizer updates:", total_training_updates)
print("Training configuration ready.")


# ==================================================
# 2. CHECKPOINT FUNCTIONS
# ==================================================

def save_checkpoint(
    path,
    epoch,
    batch_index,
    global_step,
    best_macro_f1,
    patience_counter,
    history
):

    checkpoint_data = {
        "epoch": epoch,
        "batch_index": batch_index,
        "global_step": global_step,
        "model_state_dict": e3_model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(),
        "scheduler_state_dict": scheduler.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "best_macro_f1": best_macro_f1,
        "patience_counter": patience_counter,
        "history": history
    }

    temporary_path = path + ".tmp"

    torch.save(checkpoint_data, temporary_path)
    os.replace(temporary_path, path)


def save_best_model(epoch, macro_f1):

    torch.save(
        {
            "epoch": epoch,
            "macro_f1": macro_f1,
            "model_state_dict": e3_model.state_dict(),
            "model_name": "bert-base-uncased",
            "max_length": MAX_LENGTH,
            "label_names": LABEL_NAMES,
            "random_seed": SEED,
            "architecture": "shared_bert_context_fusion",
            "fusion_dimension": 3072
        },
        BEST_MODEL_PATH
    )


# ==================================================
# 3. VALIDATION FUNCTION
# ==================================================

def evaluate_model(model, data_loader):

    model.eval()

    all_predictions = []
    all_labels = []

    with torch.inference_mode():

        for batch in data_loader:

            current_ids = batch[
                "current_input_ids"
            ].to(DEVICE)

            current_mask = batch[
                "current_attention_mask"
            ].to(DEVICE)

            parent_ids = batch[
                "parent_input_ids"
            ].to(DEVICE)

            parent_mask = batch[
                "parent_attention_mask"
            ].to(DEVICE)

            labels = batch["labels"].numpy()

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):

                logits = model(
                    current_ids,
                    current_mask,
                    parent_ids,
                    parent_mask
                )

            probabilities = torch.sigmoid(logits)

            predictions = (
                probabilities >= 0.5
            ).int()

            all_predictions.append(
                predictions.cpu().numpy()
            )

            all_labels.append(labels)

    all_predictions = np.concatenate(all_predictions)
    all_labels = np.concatenate(all_labels)

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    micro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="micro",
        zero_division=0
    )

    per_label_f1 = f1_score(
        all_labels,
        all_predictions,
        average=None,
        zero_division=0
    )

    return {
        "macro_f1": float(macro_f1),
        "micro_f1": float(micro_f1),
        "per_label_f1": per_label_f1.tolist()
    }


# ==================================================
# 4. TRAINING FUNCTION
# ==================================================

def train_one_epoch(epoch, global_step):

    e3_model.train()

    total_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    epoch_start = time.time()

    for batch_index, batch in enumerate(train_loader):

        current_ids = batch[
            "current_input_ids"
        ].to(DEVICE)

        current_mask = batch[
            "current_attention_mask"
        ].to(DEVICE)

        parent_ids = batch[
            "parent_input_ids"
        ].to(DEVICE)

        parent_mask = batch[
            "parent_attention_mask"
        ].to(DEVICE)

        labels = batch["labels"].to(DEVICE)

        # Handle final incomplete accumulation group correctly
        if batch_index == len(train_loader) - 1:
            accumulation_group_size = (
                len(train_loader) % ACCUMULATION
            )
            if accumulation_group_size == 0:
                accumulation_group_size = ACCUMULATION
        else:
            accumulation_group_size = ACCUMULATION

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):

            logits = e3_model(
                current_ids,
                current_mask,
                parent_ids,
                parent_mask
            )

            loss = criterion(logits, labels)

            scaled_loss = loss / accumulation_group_size

        scaler.scale(scaled_loss).backward()

        total_loss += loss.item()

        is_accumulation_step = (
            (batch_index + 1) % ACCUMULATION == 0
        )

        is_last_batch = (
            batch_index + 1 == len(train_loader)
        )

        if is_accumulation_step or is_last_batch:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                e3_model.parameters(),
                GRAD_CLIP
            )

            scaler.step(optimizer)
            scaler.update()

            optimizer.zero_grad(set_to_none=True)

            scheduler.step()

            global_step += 1

            # Save recovery checkpoint periodically
            if global_step % SAVE_EVERY_UPDATES == 0:

                save_checkpoint(
                    LATEST_PATH,
                    epoch,
                    batch_index,
                    global_step,
                    best_macro_f1,
                    patience_counter,
                    history
                )

        if (batch_index + 1) % 100 == 0:

            print(
                f"Epoch {epoch} | "
                f"Batch {batch_index + 1}/{len(train_loader)} | "
                f"Loss: {loss.item():.4f} | "
                f"Step: {global_step}"
            )

    average_loss = total_loss / len(train_loader)

    elapsed = time.time() - epoch_start

    return average_loss, global_step, elapsed


# ==================================================
# 5. INITIALIZE TRAINING STATE
# ==================================================

start_epoch = 1
global_step = 0
best_macro_f1 = -1.0
patience_counter = 0
history = []

print("\nStarting E3 training...")


# ==================================================
# 6. MAIN TRAINING LOOP
# ==================================================

for epoch in range(start_epoch, EPOCHS + 1):

    print("\n" + "=" * 60)
    print(f"STARTING EPOCH {epoch}/{EPOCHS}")
    print("=" * 60)

    train_loss, global_step, elapsed = train_one_epoch(
        epoch,
        global_step
    )

    print("\nRunning validation...")

    validation_results = evaluate_model(
        e3_model,
        dev_loader
    )

    val_macro_f1 = validation_results["macro_f1"]

    epoch_results = {
        "epoch": epoch,
        "train_loss": float(train_loss),
        "dev_macro_f1": val_macro_f1,
        "dev_micro_f1": validation_results["micro_f1"],
        "dev_per_label_f1": validation_results["per_label_f1"],
        "elapsed_seconds": elapsed
    }

    history.append(epoch_results)

    print("\nEPOCH RESULTS")
    print("Training loss:", round(train_loss, 4))
    print("Dev Macro-F1:", round(val_macro_f1, 4))
    print("Dev Micro-F1:", round(
        validation_results["micro_f1"], 4
    ))
    print("Time:", round(elapsed / 60, 2), "minutes")

    # Save history
    with open(HISTORY_PATH, "w") as f:
        json.dump(history, f, indent=4)

    # Best model selection
    if val_macro_f1 > best_macro_f1:

        best_macro_f1 = val_macro_f1
        patience_counter = 0

        save_best_model(epoch, val_macro_f1)

        print("New best model saved!")

    else:

        patience_counter += 1

        print("No improvement.")
        print("Patience:", patience_counter, "/", PATIENCE)

    # Save latest checkpoint after every epoch
    save_checkpoint(
        LATEST_PATH,
        epoch + 1,
        -1,
        global_step,
        best_macro_f1,
        patience_counter,
        history
    )

    print("Recovery checkpoint saved.")

    # Early stopping
    if patience_counter >= PATIENCE:

        print("Early stopping triggered.")
        break


print("\n" + "=" * 60)
print("E3 TRAINING FINISHED")
print("=" * 60)

print("Best Dev Macro-F1:", best_macro_f1)
print("Best model:", BEST_MODEL_PATH)
print("Latest checkpoint:", LATEST_PATH)


Updates per epoch: 947
Total optimizer updates: 2841
Training configuration ready.

Starting E3 training...

STARTING EPOCH 1/3
Epoch 1 | Batch 100/1893 | Loss: 0.6077 | Step: 50
Epoch 1 | Batch 200/1893 | Loss: 0.3691 | Step: 100
Epoch 1 | Batch 300/1893 | Loss: 0.2514 | Step: 150
Epoch 1 | Batch 400/1893 | Loss: 0.2252 | Step: 200
Epoch 1 | Batch 500/1893 | Loss: 0.1700 | Step: 250
Epoch 1 | Batch 600/1893 | Loss: 0.0574 | Step: 300
Epoch 1 | Batch 700/1893 | Loss: 0.0785 | Step: 350
Epoch 1 | Batch 800/1893 | Loss: 0.4904 | Step: 400
Epoch 1 | Batch 900/1893 | Loss: 0.3833 | Step: 450
Epoch 1 | Batch 1000/1893 | Loss: 0.1034 | Step: 500
Epoch 1 | Batch 1100/1893 | Loss: 0.0107 | Step: 550
Epoch 1 | Batch 1200/1893 | Loss: 0.4635 | Step: 600
Epoch 1 | Batch 1300/1893 | Loss: 0.0111 | Step: 650
Epoch 1 | Batch 1400/1893 | Loss: 0.8202 | Step: 700
Epoch 1 | Batch 1500/1893 | Loss: 0.0067 | Step: 750
Epoch 1 | Batch 1600/1893 | Loss: 0.0361 | Step: 800
Epoch 1 | Batch 1700/1893 | Loss: 

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
import os

BASE_DIR = "/content/drive/MyDrive/NLP_Research"

paths = [
    f"{BASE_DIR}/checkpoints/e3",
    f"{BASE_DIR}/reports/e3"
]

for path in paths:
    print(f"\nChecking: {path}")
    if os.path.exists(path):
        for file in os.listdir(path):
            full_path = os.path.join(path, file)
            size_mb = os.path.getsize(full_path) / (1024 * 1024)
            print(f"{file} — {size_mb:.2f} MB")
    else:
        print("Directory does not exist.")


Checking: /content/drive/MyDrive/NLP_Research/checkpoints/e3
latest_checkpoint.pt — 0.00 MB

Checking: /content/drive/MyDrive/NLP_Research/reports/e3
training_history.json — 0.00 MB


In [ ]:
import os
import torch
import json

BASE_DIR = "/content/drive/MyDrive/NLP_Research"

files = {
    "Best model": f"{BASE_DIR}/checkpoints/e3/best_model.pt",
    "Latest checkpoint": f"{BASE_DIR}/checkpoints/e3/latest_checkpoint.pt",
    "Training history": f"{BASE_DIR}/reports/e3/training_history.json"
}

for name, path in files.items():
    print(f"\n{name}")
    if not os.path.exists(path):
        print("NOT FOUND")
        continue

    size = os.path.getsize(path) / (1024 * 1024)
    print(f"Size: {size:.2f} MB")

    if size == 0:
        print("WARNING: Empty file")


Best model
NOT FOUND

Latest checkpoint
Size: 0.00 MB

Training history
Size: 0.00 MB


In [ ]:
best_path = files["Best model"]

if os.path.exists(best_path) and os.path.getsize(best_path) > 0:
    try:
        checkpoint = torch.load(
            best_path,
            map_location="cpu",
            weights_only=False
        )
        print("Best model loaded successfully!")
        print("Checkpoint type:", type(checkpoint))

        if isinstance(checkpoint, dict):
            print("Available keys:", list(checkpoint.keys()))

    except Exception as e:
        print("Checkpoint loading failed:", repr(e))
else:
    print("Best model is missing or empty.")

Best model is missing or empty.


In [ ]:
import os
import torch

encoder_path = (
    "/content/drive/MyDrive/NLP_Research/"
    "checkpoints/e2_bert_encoder.pt"
)

print("Encoder exists:", os.path.exists(encoder_path))

if os.path.exists(encoder_path):
    print(
        "Encoder size:",
        round(os.path.getsize(encoder_path) / (1024**2), 2),
        "MB"
    )

    encoder_state = torch.load(
        encoder_path,
        map_location="cpu",
        weights_only=True
    )

    print("Number of parameter tensors:", len(encoder_state))
    print("First few keys:", list(encoder_state.keys())[:5])

Encoder exists: True
Encoder size: 417.72 MB
Number of parameter tensors: 199
First few keys: ['embeddings.word_embeddings.weight', 'embeddings.position_embeddings.weight', 'embeddings.token_type_embeddings.weight', 'embeddings.LayerNorm.weight', 'embeddings.LayerNorm.bias']


In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

BASE_DIR = "/content/drive/MyDrive/NLP_Research"

encoder_path = f"{BASE_DIR}/checkpoints/e2_bert_encoder.pt"

# Load saved E2 encoder weights
encoder_state = torch.load(
    encoder_path,
    map_location="cpu",
    weights_only=True
)

# Reconstruct BERT without the pooler
shared_encoder = AutoModel.from_pretrained(
    "bert-base-uncased",
    add_pooling_layer=False
)

# Load the fine-tuned E2 weights
load_result = shared_encoder.load_state_dict(
    encoder_state,
    strict=True
)

shared_encoder = shared_encoder.to(DEVICE)

print("E2 encoder loaded successfully!")
print("Device:", DEVICE)
print("Missing keys:", load_result.missing_keys)
print("Unexpected keys:", load_result.unexpected_keys)
print("Hidden size:", shared_encoder.config.hidden_size)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
bert.pooler.dense.bias                     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
bert.pooler.dense.weight                   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


RuntimeError: Error(s) in loading state_dict for BertModel:
	Unexpected key(s) in state_dict: "pooler.dense.weight", "pooler.dense.bias". 

In [ ]:
class ContextAwareBERT(nn.Module):

    def __init__(
        self,
        shared_encoder,
        hidden_size=768,
        num_labels=5,
        fusion_dropout=0.2
    ):
        super().__init__()

        self.bert = shared_encoder

        self.classifier = nn.Sequential(
            nn.Linear(hidden_size * 4, 256),
            nn.ReLU(),
            nn.Dropout(fusion_dropout),
            nn.Linear(256, num_labels)
        )

    def encode(self, input_ids, attention_mask):

        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        return outputs.last_hidden_state[:, 0, :]

    def forward(
        self,
        current_input_ids,
        current_attention_mask,
        parent_input_ids,
        parent_attention_mask
    ):

        h_current = self.encode(
            current_input_ids,
            current_attention_mask
        )

        h_parent = self.encode(
            parent_input_ids,
            parent_attention_mask
        )

        difference = torch.abs(h_current - h_parent)
        product = h_current * h_parent

        fused = torch.cat(
            [h_current, h_parent, difference, product],
            dim=1
        )

        return self.classifier(fused)


e3_model = ContextAwareBERT(
    shared_encoder=shared_encoder,
    hidden_size=768,
    num_labels=5
).to(DEVICE)

print("E3 model reconstructed!")
print("Total parameters:", sum(p.numel() for p in e3_model.parameters()))
print("Fusion dimensions:", e3_model.classifier[0].in_features)

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: Tesla T4


In [3]:
import os

BASE_DIR = "/content/drive/MyDrive/NLP_Research"

important_files = [
    "checkpoints/e2_bert_encoder.pt",
    "datasets/e3_pairs/e3_train_pairs.csv",
    "datasets/e3_pairs/e3_dev_pairs.csv",
    "datasets/e3_pairs/e3_test_pairs.csv",
    "reports/e2_subset/e2_subset_thresholds.json",
    "reports/e2_subset/e2_subset_metrics.json"
]

for file in important_files:
    path = os.path.join(BASE_DIR, file)

    if os.path.exists(path):
        size = os.path.getsize(path) / (1024 * 1024)
        print(f"FOUND: {file} ({size:.2f} MB)")
    else:
        print(f"MISSING: {file}")

FOUND: checkpoints/e2_bert_encoder.pt (417.72 MB)
FOUND: datasets/e3_pairs/e3_train_pairs.csv (3.18 MB)
FOUND: datasets/e3_pairs/e3_dev_pairs.csv (1.13 MB)
FOUND: datasets/e3_pairs/e3_test_pairs.csv (1.35 MB)
FOUND: reports/e2_subset/e2_subset_thresholds.json (0.00 MB)
FOUND: reports/e2_subset/e2_subset_metrics.json (0.00 MB)


In [4]:
import os
import json
import torch

BASE_DIR = "/content/drive/MyDrive/NLP_Research"

# Verify E2 evaluation files
json_files = [
    f"{BASE_DIR}/reports/e2_subset/e2_subset_thresholds.json",
    f"{BASE_DIR}/reports/e2_subset/e2_subset_metrics.json"
]

for path in json_files:
    print("\nFile:", os.path.basename(path))
    print("Size in bytes:", os.path.getsize(path))

    with open(path, "r") as f:
        data = json.load(f)

    print(json.dumps(data, indent=2))

# Verify GPU
print("\n--- GPU VERIFICATION ---")
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("GPU memory:", round(
        torch.cuda.get_device_properties(0).total_memory / 1024**3, 2
    ), "GB")


File: e2_subset_thresholds.json
Size in bytes: 186
{
  "Neutral": 0.11000000000000001,
  "IdentityDirectedAbuse": 0.13,
  "AffiliationDirectedAbuse": 0.14,
  "PersonDirectedAbuse": 0.09000000000000001,
  "CounterSpeech": 0.07
}

File: e2_subset_metrics.json
Size in bytes: 794
{
  "experiment": "E2_on_E3_subset",
  "test_records": 2875,
  "thresholds": {
    "Neutral": 0.11000000000000001,
    "IdentityDirectedAbuse": 0.13,
    "AffiliationDirectedAbuse": 0.14,
    "PersonDirectedAbuse": 0.09000000000000001,
    "CounterSpeech": 0.07
  },
  "macro_f1": 0.44126703837050324,
  "micro_f1": 0.824181196024116,
  "macro_precision": 0.42803961863154016,
  "macro_recall": 0.45781534402044927,
  "hamming_loss": 0.07506086956521739,
  "subset_accuracy": 0.7770434782608696,
  "per_label_f1": {
    "Neutral": 0.9140469558296856,
    "IdentityDirectedAbuse": 0.45955882352941174,
    "AffiliationDirectedAbuse": 0.3930131004366812,
    "PersonDirectedAbuse": 0.4397163120567376,
    "CounterSpeech": 0

In [5]:
!pip -q install transformers scikit-learn pandas

In [6]:
import os
import json
import ast
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import f1_score

BASE_DIR = "/content/drive/MyDrive/NLP_Research"

DATA_DIR = f"{BASE_DIR}/datasets/e3_pairs"
CHECKPOINT_DIR = f"{BASE_DIR}/checkpoints/e3"
REPORT_DIR = f"{BASE_DIR}/reports/e3"

os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

DEVICE = torch.device("cuda")

SEED = 42
MAX_LENGTH = 256
BATCH_SIZE = 4
GRAD_ACCUMULATION = 2
EPOCHS = 3
LEARNING_RATE = 2e-5

LABEL_NAMES = [
    "Neutral",
    "IdentityDirectedAbuse",
    "AffiliationDirectedAbuse",
    "PersonDirectedAbuse",
    "CounterSpeech"
]

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

print("Environment prepared.")
print("GPU:", torch.cuda.get_device_name(0))
print("Checkpoint directory:", CHECKPOINT_DIR)

Environment prepared.
GPU: Tesla T4
Checkpoint directory: /content/drive/MyDrive/NLP_Research/checkpoints/e3


In [7]:
train_df = pd.read_csv(
    f"{DATA_DIR}/e3_train_pairs.csv",
    keep_default_na=False
)

dev_df = pd.read_csv(
    f"{DATA_DIR}/e3_dev_pairs.csv",
    keep_default_na=False
)

test_df = pd.read_csv(
    f"{DATA_DIR}/e3_test_pairs.csv",
    keep_default_na=False
)

print("Train:", train_df.shape)
print("Dev:", dev_df.shape)
print("Test:", test_df.shape)

print("\nColumns:")
print(train_df.columns.tolist())

print("\nSample:")
display(train_df[["text", "parent_text", "label_vector"]].head(2))

Train: (7571, 12)
Dev: (2390, 12)
Test: (2875, 12)

Columns:
['id', 'text', 'parent_id', 'parent_text', 'labels', 'label_vector', 'label_0', 'label_1', 'label_2', 'label_3', 'label_4', 'split']

Sample:


,text,parent_text,label_vector
0,"No, I like binging the big bang theory instead...",Literally three days old. No new developments ...,"[1, 0, 0, 0, 0]"
1,Was Michelangelo straight though? [linebreak] ...,"What's that, the Sistine Chapel? Never heard o...","[0, 1, 0, 0, 0]"


In [9]:
import torch
from transformers import AutoModel

MODEL_NAME = "bert-base-uncased"

encoder_path = (
    "/content/drive/MyDrive/NLP_Research/"
    "checkpoints/e2_bert_encoder.pt"
)

# Load E2 fine-tuned encoder weights
encoder_state = torch.load(
    encoder_path,
    map_location="cpu",
    weights_only=True
)

# IMPORTANT: Enable the pooler to match the saved checkpoint
shared_encoder = AutoModel.from_pretrained(
    MODEL_NAME,
    add_pooling_layer=True
)

# Strict loading verifies that all checkpoint keys match
result = shared_encoder.load_state_dict(
    encoder_state,
    strict=True
)

shared_encoder = shared_encoder.to(DEVICE)

print("E2 encoder loaded successfully!")
print("Missing keys:", result.missing_keys)
print("Unexpected keys:", result.unexpected_keys)
print("Hidden size:", shared_encoder.config.hidden_size)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


E2 encoder loaded successfully!
Missing keys: []
Unexpected keys: []
Hidden size: 768


In [10]:
class ContextAwareBERT(nn.Module):

    def __init__(self, shared_encoder):
        super().__init__()

        self.bert = shared_encoder

        self.classifier = nn.Sequential(
            nn.Linear(768 * 4, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 5)
        )

    def encode(self, input_ids, attention_mask):
        outputs = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        return outputs.last_hidden_state[:, 0, :]

    def forward(
        self,
        current_input_ids,
        current_attention_mask,
        parent_input_ids,
        parent_attention_mask
    ):
        h_current = self.encode(
            current_input_ids,
            current_attention_mask
        )

        h_parent = self.encode(
            parent_input_ids,
            parent_attention_mask
        )

        difference = torch.abs(h_current - h_parent)
        product = h_current * h_parent

        fused = torch.cat(
            [h_current, h_parent, difference, product],
            dim=1
        )

        return self.classifier(fused)


e3_model = ContextAwareBERT(shared_encoder).to(DEVICE)

print("E3 reconstructed.")
print("Parameters:", sum(p.numel() for p in e3_model.parameters()))

E3 reconstructed.
Parameters: 110270213


In [11]:
class ContextDataset(Dataset):

    def __init__(self, dataframe):
        self.df = dataframe.reset_index(drop=True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        labels = ast.literal_eval(row["label_vector"])

        return {
            "text": str(row["text"]),
            "parent_text": str(row["parent_text"]),
            "labels": torch.tensor(labels, dtype=torch.float32)
        }


def context_collate_fn(batch):

    current_texts = [x["text"] for x in batch]
    parent_texts = [x["parent_text"] for x in batch]
    labels = torch.stack([x["labels"] for x in batch])

    current = tokenizer(
        current_texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    parent = tokenizer(
        parent_texts,
        padding=True,
        truncation=True,
        max_length=256,
        return_tensors="pt"
    )

    return {
        "current_input_ids": current["input_ids"],
        "current_attention_mask": current["attention_mask"],
        "parent_input_ids": parent["input_ids"],
        "parent_attention_mask": parent["attention_mask"],
        "labels": labels
    }


train_loader = DataLoader(
    ContextDataset(train_df),
    batch_size=4,
    shuffle=True,
    collate_fn=context_collate_fn,
    num_workers=0,
    pin_memory=True
)

dev_loader = DataLoader(
    ContextDataset(dev_df),
    batch_size=8,
    shuffle=False,
    collate_fn=context_collate_fn,
    num_workers=0,
    pin_memory=True
)

test_loader = DataLoader(
    ContextDataset(test_df),
    batch_size=8,
    shuffle=False,
    collate_fn=context_collate_fn,
    num_workers=0,
    pin_memory=True
)

print("Train records:", len(train_loader.dataset))
print("Dev records:", len(dev_loader.dataset))
print("Test records:", len(test_loader.dataset))
print("Train batches:", len(train_loader))

Train records: 7571
Dev records: 2390
Test records: 2875
Train batches: 1893


In [12]:
batch = next(iter(train_loader))

e3_model.eval()

with torch.no_grad(), torch.autocast(
    device_type="cuda",
    dtype=torch.float16
):
    output = e3_model(
        batch["current_input_ids"].to(DEVICE),
        batch["current_attention_mask"].to(DEVICE),
        batch["parent_input_ids"].to(DEVICE),
        batch["parent_attention_mask"].to(DEVICE)
    )

print("Output shape:", output.shape)
print("Expected:", (4, 5))
print("GPU:", torch.cuda.get_device_name(0))
print("Allocated GPU memory:",
      round(torch.cuda.memory_allocated() / 1024**3, 2), "GB")

Output shape: torch.Size([4, 5])
Expected: (4, 5)
GPU: Tesla T4
Allocated GPU memory: 0.42 GB


In [13]:
import os
import torch

LATEST_PATH = f"{CHECKPOINT_DIR}/latest_model.pt"
BEST_PATH = f"{CHECKPOINT_DIR}/best_model.pt"
HISTORY_PATH = f"{REPORT_DIR}/training_history.json"

def atomic_save_model(path, model):
    temp_path = path + ".tmp"

    torch.save(
        {
            "model_state_dict": model.state_dict(),
            "epoch": current_epoch,
            "global_step": global_step
        },
        temp_path
    )

    # Verify the temporary checkpoint before replacing the old one
    check = torch.load(
        temp_path,
        map_location="cpu",
        weights_only=False
    )

    assert "model_state_dict" in check
    assert len(check["model_state_dict"]) > 0

    os.replace(temp_path, path)

    # Verify the final file
    assert os.path.getsize(path) > 0

    print(
        f"Checkpoint verified: {os.path.basename(path)} | "
        f"{os.path.getsize(path)/(1024**2):.1f} MB"
    )


# Test writing and reading a checkpoint
current_epoch = 0
global_step = 0

atomic_save_model(
    f"{CHECKPOINT_DIR}/checkpoint_test.pt",
    e3_model
)

test_checkpoint = torch.load(
    f"{CHECKPOINT_DIR}/checkpoint_test.pt",
    map_location="cpu",
    weights_only=False
)

print("Checkpoint write/read test passed.")

Checkpoint verified: checkpoint_test.pt | 420.7 MB
Checkpoint write/read test passed.


In [14]:
import math
import json
import time
import numpy as np
from sklearn.metrics import f1_score

EPOCHS = 3
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01
GRAD_ACCUMULATION = 2
SAVE_EVERY_STEPS = 250

criterion = nn.BCEWithLogitsLoss()

optimizer = torch.optim.AdamW(
    e3_model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY
)

total_optimizer_steps = math.ceil(
    len(train_loader) / GRAD_ACCUMULATION
) * EPOCHS

warmup_steps = int(0.1 * total_optimizer_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_optimizer_steps
)

scaler = torch.amp.GradScaler("cuda")

current_epoch = 0
global_step = 0
best_macro_f1 = -1
history = []

print("Training configuration ready.")
print("Optimizer steps per epoch:",
      math.ceil(len(train_loader) / GRAD_ACCUMULATION))
print("Total planned optimizer steps:", total_optimizer_steps)
print("Warmup steps:", warmup_steps)

Training configuration ready.
Optimizer steps per epoch: 947
Total planned optimizer steps: 2841
Warmup steps: 284


In [15]:
def evaluate_model(model, loader):

    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for batch in loader:

            current_ids = batch["current_input_ids"].to(DEVICE)
            current_mask = batch["current_attention_mask"].to(DEVICE)
            parent_ids = batch["parent_input_ids"].to(DEVICE)
            parent_mask = batch["parent_attention_mask"].to(DEVICE)
            labels = batch["labels"].cpu().numpy()

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(
                    current_ids,
                    current_mask,
                    parent_ids,
                    parent_mask
                )

            probabilities = torch.sigmoid(logits.float()).cpu().numpy()

            all_labels.append(labels)
            all_probs.append(probabilities)

    y_true = np.vstack(all_labels)
    y_probs = np.vstack(all_probs)
    y_pred = (y_probs >= 0.5).astype(int)

    macro = f1_score(
        y_true, y_pred, average="macro", zero_division=0
    )

    micro = f1_score(
        y_true, y_pred, average="micro", zero_division=0
    )

    return macro, micro, y_true, y_probs

In [16]:
import gc

for epoch in range(current_epoch + 1, EPOCHS + 1):

    current_epoch = epoch
    e3_model.train()

    running_loss = 0.0
    epoch_start = time.time()

    optimizer.zero_grad(set_to_none=True)

    print(f"\n========== EPOCH {epoch}/{EPOCHS} ==========")

    for batch_idx, batch in enumerate(train_loader):

        current_ids = batch["current_input_ids"].to(DEVICE)
        current_mask = batch["current_attention_mask"].to(DEVICE)
        parent_ids = batch["parent_input_ids"].to(DEVICE)
        parent_mask = batch["parent_attention_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.autocast(
            device_type="cuda",
            dtype=torch.float16
        ):
            logits = e3_model(
                current_ids,
                current_mask,
                parent_ids,
                parent_mask
            )

            loss = criterion(logits, labels)
            loss = loss / GRAD_ACCUMULATION

        scaler.scale(loss).backward()

        running_loss += loss.item() * GRAD_ACCUMULATION

        should_update = (
            (batch_idx + 1) % GRAD_ACCUMULATION == 0
            or (batch_idx + 1) == len(train_loader)
        )

        if should_update:

            scaler.unscale_(optimizer)

            torch.nn.utils.clip_grad_norm_(
                e3_model.parameters(),
                max_norm=1.0
            )

            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            global_step += 1

            if global_step % SAVE_EVERY_STEPS == 0:

                atomic_save_model(
                    LATEST_PATH,
                    e3_model
                )

                print(
                    f"Saved recovery checkpoint at step {global_step}"
                )

        if (batch_idx + 1) % 100 == 0:

            avg_loss = running_loss / (batch_idx + 1)

            print(
                f"Epoch {epoch} | "
                f"Batch {batch_idx+1}/{len(train_loader)} | "
                f"Loss: {avg_loss:.4f} | "
                f"Step: {global_step}"
            )

    epoch_loss = running_loss / len(train_loader)

    print(f"\nEpoch {epoch} training loss: {epoch_loss:.4f}")
    print("Running development evaluation...")

    dev_macro, dev_micro, _, _ = evaluate_model(
        e3_model,
        dev_loader
    )

    elapsed = (time.time() - epoch_start) / 60

    record = {
        "epoch": epoch,
        "train_loss": epoch_loss,
        "dev_macro_f1": float(dev_macro),
        "dev_micro_f1": float(dev_micro),
        "duration_minutes": elapsed,
        "global_step": global_step
    }

    history.append(record)

    print(f"Dev Macro-F1: {dev_macro:.4f}")
    print(f"Dev Micro-F1: {dev_micro:.4f}")
    print(f"Epoch duration: {elapsed:.2f} minutes")

    # Save the latest model at the end of every epoch
    atomic_save_model(LATEST_PATH, e3_model)

    # Save best model
    if dev_macro > best_macro_f1:

        best_macro_f1 = dev_macro

        atomic_save_model(BEST_PATH, e3_model)

        print("New best E3 model saved.")

    # Save history
    with open(HISTORY_PATH + ".tmp", "w") as f:
        json.dump(history, f, indent=2)

    os.replace(HISTORY_PATH + ".tmp", HISTORY_PATH)

    print("Epoch results saved to Drive.")

    gc.collect()
    torch.cuda.empty_cache()

print("\nE3 TRAINING FINISHED.")
print("Best development Macro-F1:", best_macro_f1)


========== EPOCH 1/3 ==========
Epoch 1 | Batch 100/1893 | Loss: 0.6685 | Step: 50
Epoch 1 | Batch 200/1893 | Loss: 0.5719 | Step: 100
Epoch 1 | Batch 300/1893 | Loss: 0.4784 | Step: 150
Epoch 1 | Batch 400/1893 | Loss: 0.4126 | Step: 200
Checkpoint verified: latest_model.pt | 420.7 MB
Saved recovery checkpoint at step 250
Epoch 1 | Batch 500/1893 | Loss: 0.3637 | Step: 250
Epoch 1 | Batch 600/1893 | Loss: 0.3282 | Step: 300
Epoch 1 | Batch 700/1893 | Loss: 0.2985 | Step: 350
Epoch 1 | Batch 800/1893 | Loss: 0.2775 | Step: 400
Epoch 1 | Batch 900/1893 | Loss: 0.2610 | Step: 450
Checkpoint verified: latest_model.pt | 420.7 MB
Saved recovery checkpoint at step 500
Epoch 1 | Batch 1000/1893 | Loss: 0.2509 | Step: 500
Epoch 1 | Batch 1100/1893 | Loss: 0.2393 | Step: 550
Epoch 1 | Batch 1200/1893 | Loss: 0.2282 | Step: 600
Epoch 1 | Batch 1300/1893 | Loss: 0.2196 | Step: 650
Epoch 1 | Batch 1400/1893 | Loss: 0.2152 | Step: 700
Checkpoint verified: latest_model.pt | 420.7 MB
Saved recovery 

In [17]:
print("Best checkpoint exists:", os.path.exists(BEST_PATH))
print("Latest checkpoint exists:", os.path.exists(LATEST_PATH))

if os.path.exists(BEST_PATH):
    print(
        "Best model size:",
        round(os.path.getsize(BEST_PATH) / (1024**2), 2),
        "MB"
    )

print("\nTraining history:")
print(json.dumps(history, indent=2))

Best checkpoint exists: True
Latest checkpoint exists: True
Best model size: 420.73 MB

Training history:
[
  {
    "epoch": 1,
    "train_loss": 0.18938275883776198,
    "dev_macro_f1": 0.4318693444657047,
    "dev_micro_f1": 0.8196925633568758,
    "duration_minutes": 3.716856598854065,
    "global_step": 947
  },
  {
    "epoch": 2,
    "train_loss": 0.08163587367644251,
    "dev_macro_f1": 0.43965630777030873,
    "dev_micro_f1": 0.8271425606972401,
    "duration_minutes": 4.195029179255168,
    "global_step": 1894
  },
  {
    "epoch": 3,
    "train_loss": 0.05125284229710991,
    "dev_macro_f1": 0.44052220076331955,
    "dev_micro_f1": 0.8257528556593977,
    "duration_minutes": 4.47268039782842,
    "global_step": 2841
  }
]


In [18]:
import torch
import os

BEST_PATH = f"{CHECKPOINT_DIR}/best_model.pt"

checkpoint = torch.load(
    BEST_PATH,
    map_location="cpu",
    weights_only=False
)

e3_model.load_state_dict(
    checkpoint["model_state_dict"],
    strict=True
)

e3_model = e3_model.to(DEVICE)
e3_model.eval()

print("Best E3 checkpoint loaded.")
print("Saved epoch:", checkpoint["epoch"])
print("Saved global step:", checkpoint["global_step"])
print("Checkpoint size:",
      round(os.path.getsize(BEST_PATH) / (1024**2), 2), "MB")

Best E3 checkpoint loaded.
Saved epoch: 3
Saved global step: 2841
Checkpoint size: 420.73 MB


In [19]:
import numpy as np
import pandas as pd
import torch

def get_predictions(model, loader):

    model.eval()

    all_labels = []
    all_probs = []

    with torch.no_grad():

        for batch in loader:

            current_ids = batch["current_input_ids"].to(DEVICE)
            current_mask = batch["current_attention_mask"].to(DEVICE)
            parent_ids = batch["parent_input_ids"].to(DEVICE)
            parent_mask = batch["parent_attention_mask"].to(DEVICE)

            with torch.autocast(
                device_type="cuda",
                dtype=torch.float16
            ):
                logits = model(
                    current_ids,
                    current_mask,
                    parent_ids,
                    parent_mask
                )

            probs = torch.sigmoid(logits.float())

            all_probs.append(probs.cpu().numpy())
            all_labels.append(batch["labels"].numpy())

    return (
        np.vstack(all_labels).astype(int),
        np.vstack(all_probs)
    )


print("Generating development predictions...")
y_dev, p_dev = get_predictions(e3_model, dev_loader)

print("Generating test predictions...")
y_test, p_test = get_predictions(e3_model, test_loader)

print("\nDevelopment shapes:", y_dev.shape, p_dev.shape)
print("Test shapes:", y_test.shape, p_test.shape)

Generating development predictions...
Generating test predictions...

Development shapes: (2390, 5) (2390, 5)
Test shapes: (2875, 5) (2875, 5)


In [20]:
from sklearn.metrics import f1_score

thresholds = {}

threshold_grid = np.arange(0.01, 1.00, 0.01)

for i, label in enumerate(LABEL_NAMES):

    best_threshold = 0.50
    best_f1 = -1

    for threshold in threshold_grid:

        predictions = (
            p_dev[:, i] >= threshold
        ).astype(int)

        score = f1_score(
            y_dev[:, i],
            predictions,
            zero_division=0
        )

        if score > best_f1:
            best_f1 = score
            best_threshold = float(threshold)

    thresholds[label] = best_threshold

    print(
        f"{label:28s} "
        f"Threshold: {best_threshold:.2f} | "
        f"Dev F1: {best_f1:.4f}"
    )

Neutral                      Threshold: 0.05 | Dev F1: 0.9109
IdentityDirectedAbuse        Threshold: 0.05 | Dev F1: 0.4475
AffiliationDirectedAbuse     Threshold: 0.08 | Dev F1: 0.5512
PersonDirectedAbuse          Threshold: 0.09 | Dev F1: 0.4411
CounterSpeech                Threshold: 0.05 | Dev F1: 0.1081


In [ ]:
with open(f"{REPORT_DIR}/e3_thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)

print("E3 thresholds saved.")

In [21]:
from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    hamming_loss,
    accuracy_score
)

threshold_values = np.array([
    thresholds[label] for label in LABEL_NAMES
])

y_test_pred = (
    p_test >= threshold_values
).astype(int)

metrics = {
    "experiment": "E3_BERT_context_aware",
    "test_records": int(len(y_test)),
    "thresholds": thresholds,
    "macro_f1": float(f1_score(
        y_test, y_test_pred,
        average="macro", zero_division=0
    )),
    "micro_f1": float(f1_score(
        y_test, y_test_pred,
        average="micro", zero_division=0
    )),
    "macro_precision": float(precision_score(
        y_test, y_test_pred,
        average="macro", zero_division=0
    )),
    "macro_recall": float(recall_score(
        y_test, y_test_pred,
        average="macro", zero_division=0
    )),
    "hamming_loss": float(hamming_loss(
        y_test, y_test_pred
    )),
    "subset_accuracy": float(accuracy_score(
        y_test, y_test_pred
    )),
    "per_label_f1": {}
}

for i, label in enumerate(LABEL_NAMES):
    metrics["per_label_f1"][label] = float(
        f1_score(
            y_test[:, i],
            y_test_pred[:, i],
            zero_division=0
        )
    )

print(json.dumps(metrics, indent=2))

with open(f"{REPORT_DIR}/e3_test_metrics.json", "w") as f:
    json.dump(metrics, f, indent=2)

print("\nFinal test metrics saved.")

{
  "experiment": "E3_BERT_context_aware",
  "test_records": 2875,
  "thresholds": {
    "Neutral": 0.05,
    "IdentityDirectedAbuse": 0.05,
    "AffiliationDirectedAbuse": 0.08,
    "PersonDirectedAbuse": 0.09,
    "CounterSpeech": 0.05
  },
  "macro_f1": 0.4463431207212511,
  "micro_f1": 0.8175301401107853,
  "macro_precision": 0.4411973646770824,
  "macro_recall": 0.4545374107843144,
  "hamming_loss": 0.07791304347826086,
  "subset_accuracy": 0.7777391304347826,
  "per_label_f1": {
    "Neutral": 0.9111599121581154,
    "IdentityDirectedAbuse": 0.43826086956521737,
    "AffiliationDirectedAbuse": 0.41739130434782606,
    "PersonDirectedAbuse": 0.41295546558704455,
    "CounterSpeech": 0.05194805194805195
  }
}

Final test metrics saved.


In [22]:
prediction_df = pd.DataFrame({
    "id": test_df["id"].values
})

for i, label in enumerate(LABEL_NAMES):
    prediction_df[f"true_{label}"] = y_test[:, i]
    prediction_df[f"prob_{label}"] = p_test[:, i]
    prediction_df[f"pred_{label}"] = y_test_pred[:, i]

prediction_df.to_csv(
    f"{REPORT_DIR}/e3_test_predictions.csv",
    index=False
)

print("Test predictions saved.")
print("Shape:", prediction_df.shape)

Test predictions saved.
Shape: (2875, 16)


In [23]:
with open(
    f"{BASE_DIR}/reports/e2_subset/e2_subset_metrics.json"
) as f:
    e2_metrics = json.load(f)

comparison = pd.DataFrame([
    {
        "Model": "E2 BERT Text-only",
        "Macro-F1": e2_metrics["macro_f1"],
        "Micro-F1": e2_metrics["micro_f1"],
        "Macro Precision": e2_metrics["macro_precision"],
        "Macro Recall": e2_metrics["macro_recall"]
    },
    {
        "Model": "E3 BERT Context-aware",
        "Macro-F1": metrics["macro_f1"],
        "Micro-F1": metrics["micro_f1"],
        "Macro Precision": metrics["macro_precision"],
        "Macro Recall": metrics["macro_recall"]
    }
])

display(comparison.round(4))

macro_difference = (
    metrics["macro_f1"] - e2_metrics["macro_f1"]
)

print(f"\nE3 - E2 Macro-F1: {macro_difference:+.4f}")
print(f"Relative change: {macro_difference / e2_metrics['macro_f1'] * 100:+.2f}%")

,Model,Macro-F1,Micro-F1,Macro Precision,Macro Recall
0,E2 BERT Text-only,0.4413,0.8242,0.4280,0.4578
1,E3 BERT Context-aware,0.4463,0.8175,0.4412,0.4545



E3 - E2 Macro-F1: +0.0051
Relative change: +1.15%


In [24]:
final_files = [
    BEST_PATH,
    LATEST_PATH,
    f"{REPORT_DIR}/training_history.json",
    f"{REPORT_DIR}/e3_thresholds.json",
    f"{REPORT_DIR}/e3_test_metrics.json",
    f"{REPORT_DIR}/e3_test_predictions.csv"
]

print("FINAL E3 ARTIFACT CHECK")

for path in final_files:
    exists = os.path.exists(path)
    size = os.path.getsize(path) if exists else 0

    print(
        ("OK" if exists and size > 0 else "MISSING/EMPTY"),
        os.path.basename(path),
        f"({size / (1024**2):.2f} MB)"
    )

FINAL E3 ARTIFACT CHECK
OK best_model.pt (420.73 MB)
OK latest_model.pt (420.73 MB)
OK training_history.json (0.00 MB)
MISSING/EMPTY e3_thresholds.json (0.00 MB)
OK e3_test_metrics.json (0.00 MB)
OK e3_test_predictions.csv (0.25 MB)


In [25]:
import os
import json

threshold_path = os.path.join(REPORT_DIR, "e3_thresholds.json")

# Save thresholds safely
temp_path = threshold_path + ".tmp"

with open(temp_path, "w") as f:
    json.dump(thresholds, f, indent=2)

os.replace(temp_path, threshold_path)

# Read back and verify
with open(threshold_path, "r") as f:
    saved_thresholds = json.load(f)

assert saved_thresholds == thresholds

print("Threshold file successfully saved and verified.")
print("Actual file size:", os.path.getsize(threshold_path), "bytes")
print(json.dumps(saved_thresholds, indent=2))

Threshold file successfully saved and verified.
Actual file size: 146 bytes
{
  "Neutral": 0.05,
  "IdentityDirectedAbuse": 0.05,
  "AffiliationDirectedAbuse": 0.08,
  "PersonDirectedAbuse": 0.09,
  "CounterSpeech": 0.05
}


In [26]:
for filename in os.listdir(REPORT_DIR):
    path = os.path.join(REPORT_DIR, filename)

    if os.path.isfile(path):
        print(
            filename,
            "|",
            os.path.getsize(path),
            "bytes"
        )

training_history.json | 635 bytes
e3_test_metrics.json | 722 bytes
e3_test_predictions.csv | 258820 bytes
e3_thresholds.json | 146 bytes


In [27]:
import os

paths = [
    f"{BASE_DIR}/checkpoints/e2_bert_encoder.pt",
    f"{BASE_DIR}/checkpoints/e3/best_model.pt",
    f"{BASE_DIR}/checkpoints/e3/latest_model.pt"
]

for path in paths:
    print(
        "SAVED" if os.path.exists(path) and os.path.getsize(path) > 0 else "MISSING",
        path,
        f"({os.path.getsize(path)/(1024**2):.2f} MB)"
        if os.path.exists(path) else ""
    )

SAVED /content/drive/MyDrive/NLP_Research/checkpoints/e2_bert_encoder.pt (417.72 MB)
SAVED /content/drive/MyDrive/NLP_Research/checkpoints/e3/best_model.pt (420.73 MB)
SAVED /content/drive/MyDrive/NLP_Research/checkpoints/e3/latest_model.pt (420.73 MB)


In [28]:
import os

BASE_DIR = "/content/drive/MyDrive/NLP_Research"

files = [
    "checkpoints/e2_bert_encoder.pt",
    "checkpoints/e3/best_model.pt",
    "checkpoints/e3/latest_model.pt",
    "reports/e3/training_history.json",
    "reports/e3/e3_thresholds.json",
    "reports/e3/e3_test_metrics.json",
    "reports/e3/e3_test_predictions.csv"
]

for f in files:
    path = os.path.join(BASE_DIR, f)
    if os.path.exists(path):
        print(f, round(os.path.getsize(path)/1024**2, 2), "MB")
    else:
        print("MISSING:", f)

checkpoints/e2_bert_encoder.pt 417.72 MB
checkpoints/e3/best_model.pt 420.73 MB
checkpoints/e3/latest_model.pt 420.73 MB
reports/e3/training_history.json 0.0 MB
reports/e3/e3_thresholds.json 0.0 MB
reports/e3/e3_test_metrics.json 0.0 MB
reports/e3/e3_test_predictions.csv 0.25 MB
